In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import gradio as gr

In [ ]:
load_dotenv(override=True)

In [ ]:
# Step 1. Defining the state object
class State(BaseModel):
        
    messages: Annotated[list, add_messages]

In [ ]:
# Step 2. Start the Graph Builder
graph_builder = StateGraph(State)

In [ ]:
# Step 3. Create a node
llm = ChatOpenAI(model="gpt-oss-20b")

def chatbot_node(old_state: State) -> State:
    reply = llm.invoke(old_state.messages)
    new_state = State(messages=[reply])
    return new_state

graph_builder.add_node("chatbot", chatbot_node)


In [ ]:
# Step 4. Create Edges
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

In [ ]:
# Step 5. Compile the Graph
graph = graph_builder.compile()

In [ ]:
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
def chat(user_input: str, history):
    initial_state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(initial_state)
    print(result)
    return result['messages'][-1].content


gr.ChatInterface(chat).launch()